# Urdu POS Tagger
Minimal Part-of-Speech tagger for Urdu using **NLTK** (Unigram → Bigram backoff chain).

**Tags (CLE style):** NN, NNP, PRP, VB, VBZ, VBD, JJ, RB, DT, IN, CC, CD, NEG, WP, PUNC

In [ ]:
# !pip install nltk scikit-learn matplotlib seaborn

In [ ]:
import nltk
from nltk.tag import UnigramTagger, BigramTagger
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

## 1. Load Data

In [ ]:
def load_corpus(filepath):
    """Load CLE corpus: WORD\tTAG per line, blank line = sentence boundary."""
    sentences, current = [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('\t')
                if len(parts) == 2:
                    current.append(tuple(parts))
            elif current:
                sentences.append(current)
                current = []
    if current:
        sentences.append(current)
    return sentences

sentences = load_corpus('data/sample_corpus.txt')
print(f"Sentences: {len(sentences)}")
print(f"Example:   {sentences[0]}")

## 2. Train / Test Split

In [ ]:
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f"Train: {len(train_data)}  |  Test: {len(test_data)}")

## 3. Train NLTK Backoff Tagger
BigramTagger backs off to UnigramTagger, which backs off to the default tag `NN`.

In [ ]:
t0 = nltk.DefaultTagger('NN')
t1 = UnigramTagger(train_data, backoff=t0)
t2 = BigramTagger(train_data, backoff=t1)

print(f"Accuracy: {t2.accuracy(test_data):.2%}")

## 4. Detailed Metrics

In [ ]:
true_tags, pred_tags = [], []
for sent in test_data:
    words     = [w for w, _ in sent]
    true_tags += [t for _, t in sent]
    pred_tags += [t for _, t in t2.tag(words)]

print(classification_report(true_tags, pred_tags, zero_division=0))

## 5. Confusion Matrix

In [ ]:
labels = sorted(set(true_tags))
cm = confusion_matrix(true_tags, pred_tags, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(pd.DataFrame(cm, index=labels, columns=labels),
            annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Tag')
plt.xlabel('Predicted Tag')
plt.tight_layout()
plt.show()

## 6. Tag a New Sentence

In [ ]:
sample = ['میں', 'نے', 'کتاب', 'پڑھی', '۔']
for word, tag in t2.tag(sample):
    print(f"{word:15s} → {tag}")